In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from data_processing import process_data
from config_loader import MODEL_PATH, TARGET_COLUMNS, TEST_PATH, RESULTS_PATH, SMILES_FILE, SEED, FINGERPRINT_NBITS, TRAINING_ITERATIONS

In [ ]:
df_train_X, df_train_y, df_test_X, df_test_y = process_data()

In [ ]:
print("train and test data shape:", df_train_X.shape, df_test_X.shape)

In [ ]:
from fp_embed import FingerprintEmbedder
from config_loader import SMILES_FILE, FINGERPRINT_NBITS

embedder = FingerprintEmbedder(smiles_path=SMILES_FILE, nBits=FINGERPRINT_NBITS)

In [ ]:
X_enc = embedder.encode_features(df_train_X)
y_enc = embedder.encode_target(df_train_y, target_column=TARGET_COLUMNS[0])
# y_enc_exp = np.exp(y_enc)

train_x = torch.tensor(X_enc.values, dtype=torch.double)
train_y = torch.tensor(y_enc[TARGET_COLUMNS[0]].values, dtype=torch.double)

In [ ]:
test_X_enc = embedder.encode_features(df_test_X)
test_y_enc = embedder.encode_target(df_test_y, target_column=TARGET_COLUMNS[0])
# test_y_enc_exp = np.exp(test_y_enc)

test_x = torch.tensor(test_X_enc.values, dtype=torch.double)
test_y = torch.tensor(test_y_enc[TARGET_COLUMNS[0]].values, dtype=torch.double)
print(test_x.shape)
print(test_y.shape)

In [ ]:
from gp_model import GPModel
from config_loader import FINGERPRINT_NBITS

gp = GPModel.from_params(noise_type='fixed', fixed_noise_value=0.05, nBits=FINGERPRINT_NBITS)
gp_model = gp.fit(train_x, train_y, train_yvar=None, use_hvarfner_prior=True)

In [ ]:
from utils.plot import cv_plots

cv_plots(gp_model, train_x, train_y)

In [ ]:
cv_plots(gp_model, test_x, test_y)

In [ ]:
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.acquisition.monte_carlo import qExpectedImprovement

acq_func = qLogExpectedImprovement(gp_model, best_f=train_y.max())

In [ ]:
acq_func(test_x.unsqueeze(1))

In [ ]:
all_candidates_df = pd.read_csv("data/all_candidates.csv")

encoded_candidates = embedder.encode_features(all_candidates_df)
candidates_X = torch.tensor(encoded_candidates.values, dtype=torch.double)
print(f"Encoded candidates shape: {candidates_X.shape}")
print(acq_func(candidates_X.unsqueeze(1)))

# Compute acquisition values for all candidates
acq_values_candidates = acq_func(candidates_X.unsqueeze(1)).squeeze(-1).detach().cpu().numpy()

with torch.no_grad():
    pred_absorbance = gp_model.posterior(candidates_X).mean.squeeze(-1).cpu().numpy()

all_candidates_df['acq_value'] = acq_values_candidates
all_candidates_df['pred_absorbance'] = pred_absorbance

all_candidates_df['n_1'] = (all_candidates_df['n_1']).round().astype(int)
all_candidates_df['n_2'] = (all_candidates_df['n_2']).round().astype(int)

top_k = 50
# already collected candidates data
collected_candidates_df = pd.read_csv("data/ligand_data.csv")
collected_candidates = collected_candidates_df[['Ligand 1', 'n_1', 'Ligand 2', 'n_2']].drop_duplicates()

# Create a set of tuples for faster lookup
collected_set = set(collected_candidates.itertuples(index=False, name=None))

# Filter out candidates that are already collected
mask = ~all_candidates_df[['Ligand 1', 'n_1', 'Ligand 2', 'n_2']].apply(tuple, axis=1).isin(collected_set)
remaining_candidates_df = all_candidates_df[mask]

print(f"Total candidates: {len(all_candidates_df)}")
print(f"Already collected candidates: {len(collected_candidates)}")
print(f"Remaining candidates after filtering: {len(remaining_candidates_df)}")

# Get top k candidates from the filtered DataFrame
top_k_df = remaining_candidates_df.nlargest(top_k, 'acq_value')[['Ligand 1', 'n_1', 'Ligand 2', 'n_2', 'acq_value', 'pred_absorbance']]

display(top_k_df)

top_k_df.to_csv("results/top_50.csv", index=False)
print(f"Top {top_k} candidates saved to results/top_50.csv")